# Two-Stage Retrieval Audit

Notebook de transparence sur le retrieval BOFIP en **deux étages**:
1. retrieval documentaire BOI
2. retrieval de passages dans les documents retenus

Portée:
- pas de LLM
- pas d'abstention encore
- focus sur la lisibilité du pipeline et le chunk ordering


In [1]:
from pathlib import Path
import json
import sys

PROJECT_ROOT = Path.cwd().resolve().parent
SRC_ROOT = PROJECT_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from bofip_cleanroom.jsonio import read_jsonl
from bofip_cleanroom.models import chunk_node_from_dict, raw_document_from_dict

def load_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))

raw_docs = [raw_document_from_dict(item) for item in read_jsonl(PROJECT_ROOT / "data" / "interim" / "raw_docs_sample_1000.jsonl")]
chunks = [chunk_node_from_dict(item) for item in read_jsonl(PROJECT_ROOT / "data" / "interim" / "chunks_section_window_sample_1000.jsonl")]
queries = read_jsonl(PROJECT_ROOT / "data" / "interim" / "retrieval_queries_sample_1000_v1.jsonl")

doc_eval = load_json(PROJECT_ROOT / "data" / "reports" / "phase3_doc_lexical_eval_raw_docs_sample_1000.json")
chunk_eval = load_json(PROJECT_ROOT / "data" / "reports" / "phase3_batch_eval_chunks_section_window_sample_1000.json")
two_stage_full = load_json(PROJECT_ROOT / "data" / "reports" / "phase3_two_stage_probe_sample_1000_full.json")
two_stage_body = load_json(PROJECT_ROOT / "data" / "reports" / "phase3_two_stage_probe_sample_1000_body.json")
chunk_order_audit = load_json(PROJECT_ROOT / "data" / "reports" / "phase3_chunk_order_audit_sample_1000.json")

query_map = {row["id"]: row for row in queries}
doc_eval_map = {row["id"]: row for row in doc_eval["results"]}
chunk_eval_map = {row["id"]: row for row in chunk_eval["results"]}
two_stage_full_map = {row["id"]: row for row in two_stage_full["rows"]}
two_stage_body_map = {row["id"]: row for row in two_stage_body["rows"]}
raw_doc_map = {doc.boi_reference: doc for doc in raw_docs}

print("Loaded two-stage audit artifacts.")


Loaded two-stage audit artifacts.


## 1. Gate actuel

Le point important à lire correctement:
- le gate `80%` a été franchi **au niveau retrieval documentaire**
- pas encore au niveau génération
- le prochain problème est la sélection du bon passage **dans** le bon document


In [2]:
{
    "doc_lexical_metrics": doc_eval["metrics"],
    "flat_chunk_lexical_metrics": chunk_eval["metrics"],
    "chunk_order_audit_summary": chunk_order_audit["summary"],
}


{'doc_lexical_metrics': {'hit@1': 0.9333, 'hit@3': 0.9778, 'hit@5': 0.9778},
 'flat_chunk_lexical_metrics': {'hit@1': 0.7333,
  'hit@3': 0.9778,
  'hit@5': 1.0},
 'chunk_order_audit_summary': {'full': {'query_count': 42,
   'title_only_section_rate': 0.8333,
   'actualite_rate': 0.1667,
   'avg_token_count': 175.4},
  'leaf': {'query_count': 42,
   'title_only_section_rate': 0.9048,
   'actualite_rate': 0.2143,
   'avg_token_count': 172.4},
  'body': {'query_count': 42,
   'title_only_section_rate': 0.619,
   'actualite_rate': 0.0476,
   'avg_token_count': 198.7}}}

## 2. Ce qui change entre stage 2 `full` et stage 2 `body`

Le stage 1 garde la structure BOI.
Le stage 2 `body` retire le bruit structurel inutile et cherche le meilleur passage sur le texte du chunk lui-même.


In [3]:
chunk_order_audit["examples_where_body_differs_from_full"][:8]


[{'id': 'q02',
  'pattern': 'near_neighbor',
  'query': "quelles dépenses de fonctionnement sont éligibles au crédit d'impôt recherche",
  'boi_reference': 'BOI-BIC-RICI-10-10-20-25-20250813',
  'full_top_chunk': {'chunk_id': '10594-PGP__section_window__10594-pgp-paragraph-00000__10594-pgp-paragraph-00001__bic-r-ductions-et-cr-dits-d-imp-t-cr-dits-d-imp-t-cr-dit-d-imp-t-recherche-d-penses-de-recherche-ligibles-autres-d-penses-de-fonctionnement',
   'score': 3.0036,
   'chunk_kind': 'paragraph',
   'token_count': 51,
   'section_path': 'BIC - Réductions et crédits d’impôt - Crédits d’impôt - Crédit d’impôt recherche - Dépenses de recherche éligibles - Autres dépenses de fonctionnement',
   'text': 'Actualité liée : [node:date:14709-PGP] : BIC - Aménagements du crédit d’impôt recherche, du crédit d’impôt collection et du crédit d’impôt innovation (loi n° 2025-127 du 14 février 2025 de finances pour 2025, art. 55, 56, 57, 58 et 77, I-2°) (1)'},
  'body_top_chunk': {'chunk_id': '10594-PGP_

In [4]:
def document_preview(boi_reference, section_limit=10):
    doc = raw_doc_map[boi_reference]
    return {
        "boi_reference": doc.boi_reference,
        "title": doc.title,
        "publication_date": doc.publication_date,
        "sections_preview": [
            {
                "level": section.level,
                "title": section.title,
                "path": section.path,
            }
            for section in doc.sections[:section_limit]
        ],
    }

document_preview("BOI-BIC-RICI-10-10-20-25-20250813")


{'boi_reference': 'BOI-BIC-RICI-10-10-20-25-20250813',
 'title': 'BIC - Réductions et crédits d’impôt - Crédits d’impôt - Crédit d’impôt recherche - Dépenses de recherche éligibles - Autres dépenses de fonctionnement',
 'publication_date': '2025-08-13',
 'sections_preview': [{'level': 0,
   'title': 'BIC - Réductions et crédits d’impôt - Crédits d’impôt - Crédit d’impôt recherche - Dépenses de recherche éligibles - Autres dépenses de fonctionnement',
   'path': ['BIC - Réductions et crédits d’impôt - Crédits d’impôt - Crédit d’impôt recherche - Dépenses de recherche éligibles - Autres dépenses de fonctionnement']},
  {'level': 1,
   'title': 'I. Principes applicables aux autres dépenses de fonctionnement',
   'path': ['I. Principes applicables aux autres dépenses de fonctionnement']},
  {'level': 1,
   'title': 'II. Précisions concernant les entreprises soumises à l’impôt sur le revenu',
   'path': ['II. Précisions concernant les entreprises soumises à l’impôt sur le revenu']}]}

## 3. Helpers d'affichage


In [5]:
def top_chunks(trace_row, limit=6):
    return [
        {
            "global_rank": hit["global_rank"],
            "document_rank": hit["document_rank"],
            "document_score": hit["document_score"],
            "local_rank": hit["local_rank"],
            "local_score": hit["local_score"],
            "boi_reference": hit["boi_reference"],
            "chunk_kind": hit["chunk_kind"],
            "section_path": hit["section_path"],
            "text": hit["text"],
        }
        for hit in trace_row["chunk_hits"][:limit]
    ]

def top_documents(trace_row, limit=3):
    return trace_row["document_hits"][:limit]

def compare_trace(query_id):
    query = query_map[query_id]
    doc_row = doc_eval_map[query_id]
    return {
        "query_id": query_id,
        "pattern": query.get("pattern"),
        "query": query["query"],
        "expected_boi": query.get("expected_boi"),
        "stage1_top_docs": top_documents(two_stage_body_map[query_id]),
        "stage2_full_top_chunks": top_chunks(two_stage_full_map[query_id]),
        "stage2_body_top_chunks": top_chunks(two_stage_body_map[query_id]),
        "stage1_doc_eval": {
            "returned_boi": doc_row["returned_boi"],
            "supported_query": doc_row.get("supported_query"),
            "hit@1": doc_row.get("hit@1"),
            "hit@3": doc_row.get("hit@3"),
            "hit@5": doc_row.get("hit@5"),
        },
    }


## 4. Exemples réels, variés

Chaque bloc montre:
- la question
- le stage 1: top documents
- le stage 2 actuel `full`
- le stage 2 passage `body`


In [6]:
compare_trace("q02")


{'query_id': 'q02',
 'pattern': 'near_neighbor',
 'query': "quelles dépenses de fonctionnement sont éligibles au crédit d'impôt recherche",
 'expected_boi': 'BOI-BIC-RICI-10-10-20-25-20250813',
 'stage1_top_docs': [{'rank': 1,
   'score': 39.793,
   'boi_reference': 'BOI-BIC-RICI-10-10-20-25-20250813',
   'title': 'BIC - Réductions et crédits d’impôt - Crédits d’impôt - Crédit d’impôt recherche - Dépenses de recherche éligibles - Autres dépenses de fonctionnement'},
  {'rank': 2,
   'score': 31.7873,
   'boi_reference': 'BOI-BIC-RICI-10-10-20-50-20250813',
   'title': 'BIC - Réductions et crédits d’impôt - Crédits d’impôt - Crédit d’impôt recherche - Dépenses de recherche éligibles - Dépenses de normalisation afférentes aux produits de l’entreprise'},
  {'rank': 3,
   'score': 29.8877,
   'boi_reference': 'BOI-BIC-RICI-10-10-20-40-20250813',
   'title': 'BIC - Réductions et crédits d’impôt - Crédits d’impôt - Crédit d’impôt recherche - Dépenses de recherche éligibles relatives aux brev

In [7]:
compare_trace("q04")


{'query_id': 'q04',
 'pattern': 'near_neighbor',
 'query': "extension du délai de reprise pour l'imposition des revenus de 2018 prélèvement à la source",
 'expected_boi': 'BOI-IR-PAS-50-20-50-20200210',
 'stage1_top_docs': [{'rank': 1,
   'score': 56.1877,
   'boi_reference': 'BOI-IR-PAS-50-20-50-20200210',
   'title': "IR - Prélèvement à la source de l'impôt sur le revenu - Mesures transitoires - Autres mesures transitoires - Extension du délai de reprise pour l'imposition des revenus de 2018"},
  {'rank': 2,
   'score': 25.9773,
   'boi_reference': 'BOI-CF-PGR-10-70-20161229',
   'title': "CF - Prescription du droit de reprise de l'administration - Prorogation du délai de reprise en cas d'activités occultes ou de procès-verbal pour flagrance fiscale et conséquences sur certains délais"},
  {'rank': 3,
   'score': 25.9532,
   'boi_reference': 'BOI-IR-PAS-20-30-30-20180515',
   'title': "IR - Prélèvement à la source de l'impôt sur le revenu - Calcul du prélèvement - Actualisation du pr

In [8]:
compare_trace("q08")


{'query_id': 'q08',
 'pattern': 'near_neighbor',
 'query': "sort des déficits de l'ancien groupe après absorption de la société mère",
 'expected_boi': 'BOI-IS-GPE-50-10-30-20210811',
 'stage1_top_docs': [{'rank': 1,
   'score': 53.2281,
   'boi_reference': 'BOI-IS-GPE-50-10-30-20210811',
   'title': "IS - Régime fiscal des groupes de sociétés - Opérations de restructurations du groupe - Absorption de la société mère, de l'entité mère non résidente ou d'une société étrangère - Sort des déficits, charges financières nettes non déduites et capacité de déduction inemployée de l'ancien groupe"},
  {'rank': 2,
   'score': 46.0748,
   'boi_reference': 'BOI-IS-GPE-50-30-30-20210811',
   'title': "IS - Régime fiscal des groupes de sociétés - Opérations de restructurations du groupe - Scission de la société mère ou de l'entité mère non résidente - Sort des déficits, charges financières non déduites et capacité de déduction inemployée de l'ancien groupe"},
  {'rank': 3,
   'score': 38.858,
   'b

In [9]:
compare_trace("q11")


{'query_id': 'q11',
 'pattern': 'rescrit_case',
 'query': 'taux de TVA pour un système qui fixe un fauteuil roulant à une trottinette électrique',
 'expected_boi': 'BOI-RES-TVA-000074-20210309',
 'stage1_top_docs': [{'rank': 1,
   'score': 60.1241,
   'boi_reference': 'BOI-RES-TVA-000074-20210309',
   'title': 'RES - Taxe sur la valeur ajoutée - Liquidation - Taux de TVA applicable aux systèmes de fixation permettant d’accrocher un fauteuil roulant à une trottinette électrique'},
  {'rank': 2,
   'score': 19.9509,
   'boi_reference': 'BOI-IR-RICI-390-20230629',
   'title': "IR - Réductions et crédits d'impôt - Crédit d’impôt pour le premier abonnement à un journal, à une publication périodique ou à un service de presse en ligne d'information politique et générale"},
  {'rank': 3,
   'score': 18.4469,
   'boi_reference': 'BOI-RES-TPS-000060-20210309',
   'title': "RES - Taxes et particpations sur les salaires - Taxe sur les salaires - Exonération applicable aux établissements d'enseigne

In [10]:
compare_trace("q12")


{'query_id': 'q12',
 'pattern': 'topic_specific',
 'query': 'détermination du redevable de la TVA pour les livraisons de biens et prestations de services',
 'expected_boi': 'BOI-TVA-DECLA-10-10-20-20251022',
 'stage1_top_docs': [{'rank': 1,
   'score': 52.6082,
   'boi_reference': 'BOI-TVA-DECLA-10-10-20-20251022',
   'title': 'TVA - Régimes d’imposition et obligations déclaratives et comptables - Redevable de la taxe - Livraisons de biens et prestations de services - Détermination du redevable'},
  {'rank': 2,
   'score': 46.868,
   'boi_reference': 'BOI-TVA-DECLA-10-10-20200323',
   'title': "TVA - Régimes d'imposition et obligations déclaratives et comptables - Redevable de la taxe - Livraisons de biens et prestations de services"},
  {'rank': 3,
   'score': 43.279,
   'boi_reference': 'BOI-TVA-DECLA-10-10-30-20-20200902',
   'title': "TVA - Régimes d'imposition et obligations déclaratives et comptables - Redevable de la taxe - Livraisons de biens et prestations de services - Solida

In [11]:
compare_trace("q14")


{'query_id': 'q14',
 'pattern': 'procedural_question',
 'query': "compensation et substitution de base légale en contentieux de l'assiette",
 'expected_boi': 'BOI-CTX-DG-20-40-10-20120912',
 'stage1_top_docs': [{'rank': 1,
   'score': 47.3713,
   'boi_reference': 'BOI-CTX-DG-20-40-10-20120912',
   'title': "CTX - Contentieux de l'assiette de l'impôt - Dispositions communes - Compensation et substitution de base légale susceptibles d'être opposées par l'administration"},
  {'rank': 2,
   'score': 22.1031,
   'boi_reference': 'BOI-CTX-DG-10-10-20120912',
   'title': "CTX - Contentieux de l'assiette de l'impôt – Nature et place du contentieux de l'assiette de l'impôt"},
  {'rank': 3,
   'score': 19.514,
   'boi_reference': 'BOI-CTX-DG-20-60-20120912',
   'title': "CTX - Contentieux de l'assiette de l'impôt - Dispositions communes - Question prioritaire de constitutionnalité et questions préjudicielles"}],
 'stage2_full_top_chunks': [{'global_rank': 1,
   'document_rank': 1,
   'document_s

In [12]:
compare_trace("q29")


{'query_id': 'q29',
 'pattern': 'explicit_reference',
 'query': 'États et territoires non coopératifs droit conventionnel',
 'expected_boi': 'BOI-INT-DG-20-50-20210224',
 'stage1_top_docs': [{'rank': 1,
   'score': 47.7398,
   'boi_reference': 'BOI-INT-DG-20-50-10-20210224',
   'title': 'INT - Dispositions communes - Droit conventionnel - États et territoires non coopératifs - Constitution et mise à jour de la liste des États et territoires non coopératifs'},
  {'rank': 2,
   'score': 47.22,
   'boi_reference': 'BOI-INT-DG-20-50-20210224',
   'title': 'INT - Dispositions communes - Droit conventionnel - États et territoires non coopératifs'},
  {'rank': 3,
   'score': 13.6365,
   'boi_reference': 'BOI-INT-DG-20-20-40-20120912',
   'title': "INT -Dispositions communes - Droit conventionnel - Modalités d'imposition au regard du droit conventionnel - Revenus immobiliers - Gains en capital - Professions indépendantes - Revenus d'emploi – Tantièmes"}],
 'stage2_full_top_chunks': [{'global_r

In [13]:
compare_trace("q30")


{'query_id': 'q30',
 'pattern': 'topic_specific',
 'query': 'prélèvement forfaitaire obligatoire applicable aux produits de placement à revenu fixe',
 'expected_boi': 'BOI-RPPM-RCM-30-20-40-20220630',
 'stage1_top_docs': [{'rank': 1,
   'score': 44.2023,
   'boi_reference': 'BOI-RPPM-RCM-30-20-70-20191220',
   'title': "RPPM - Revenus de capitaux mobiliers, gains et profits assimilés - Modalités particulières d'imposition - Prélèvement forfaitaire obligatoire non libératoire de l'impôt sur le revenu applicable aux produits de placement à revenu fixe, aux produits et gains de cession de bons ou contrats de capitalisation et placements de même nature attachés à des primes versées à compter du 27 septembre 2017, et aux revenus distribués - Mesures de contrôle applicables aux produits de placement à revenu fixe"},
  {'rank': 2,
   'score': 40.1469,
   'boi_reference': 'BOI-RPPM-RCM-30-20-40-20220630',
   'title': "RPPM - Revenus de capitaux mobiliers, gains et profits assimilés - Modalités

In [14]:
compare_trace("u04")


{'query_id': 'u04',
 'pattern': 'false_premise',
 'query': "l'article 44 sexies-0 A du CGI supprime-t-il le remboursement immédiat du CIR pour les JEI",
 'expected_boi': None,
 'stage1_top_docs': [{'rank': 1,
   'score': 23.9039,
   'boi_reference': 'BOI-RES-BIC-000062-20210309',
   'title': "RES - Bénéfices industriels et commerciaux - Champ d'application et territorialité - Zones franches d'activités situées dans les départements d'outre-mer - Éligibilité de l'activité de transport par véhicule sanitaire léger (VSL) à l'abattement prévu à l'article 44 quaterdecies du CGI"},
  {'rank': 2,
   'score': 21.2348,
   'boi_reference': 'BOI-RES-RSA-000127-20250812',
   'title': 'RES - Revenus salariaux et assimilés - Épargne salariale et actionnariat salarié - Non-éligibilité au régime du sursis d’imposition prévu à l’article 150-0 B du CGI des gains résultant de l’apport de titres souscrits en exercice de bons de souscription de parts de créateur d’entreprise (BSPCE)'},
  {'rank': 3,
   'sc

In [15]:
compare_trace("u05")


{'query_id': 'u05',
 'pattern': 'false_premise',
 'query': 'le BOFIP indique-t-il que tous les organismes sans but lucratif sont toujours exonérés de TVA',
 'expected_boi': None,
 'stage1_top_docs': [{'rank': 1,
   'score': 32.7319,
   'boi_reference': 'BOI-TVA-DECLA-20-30-20-10-20220323',
   'title': "TVA - Régimes d'imposition et obligations déclaratives et comptables - Obligations et formalités déclaratives - Obligations et formalités particulières - Mesures propres à certaines entreprises - Régime applicable aux organismes sans but lucratif"},
  {'rank': 2,
   'score': 14.3276,
   'boi_reference': 'BOI-IS-CHAMP-10-50-20-20120912',
   'title': "IS - Champ d'application et territorialité - Collectivités imposables - Organismes privés autres que les sociétés - Organismes réalisant des activités lucratives accessoires"},
  {'rank': 3,
   'score': 13.9393,
   'boi_reference': 'BOI-TVA-CHAMP-10-10-50-30-20120912',
   'title': "TVA - Champ d'application et territorialité - Opérations impo

## 5. Lecture d'ingénieur

À lire de façon stricte:
- si le stage 1 se trompe, le stage 2 ne peut pas sauver le bon document
- quand le stage 1 est correct, `body` donne souvent un passage plus substantiel que `full`
- les questions hors corpus / à fausse prémisse retournent encore des matchs de proximité: l'abstention n'est pas encore branchée


In [16]:
{
    "doc_stage_misses": [
        row for row in doc_eval["results"]
        if row["supported_query"] and not row["hit@1"]
    ],
    "unsupported_examples": [
        {
            "id": row["id"],
            "query": row["query"],
            "returned_boi": row["returned_boi"][:3],
        }
        for row in doc_eval["results"]
        if not row["supported_query"]
    ][:5],
}


{'doc_stage_misses': [{'id': 'q01',
   'pattern': 'near_neighbor',
   'query': 'une JEI peut-elle demander le remboursement immédiat du CIR',
   'expected_boi': 'BOI-BIC-CHAMP-80-20-20-20-20240703',
   'supported_query': True,
   'returned_boi': ['BOI-RES-SJ-000014-20210309',
    'BOI-TVA-SECT-80-60-10-50-20120912',
    'BOI-ENR-DG-50-20-20160203',
    'BOI-TVA-DED-50-20-10-20150506',
    'BOI-TVA-DED-50-20-20-20210224'],
   'top_hits': [{'rank': 1,
     'score': 11.7225,
     'boi_reference': 'BOI-RES-SJ-000014-20210309',
     'chunk_id': '11456-PGP__document_title',
     'chunk_kind': 'document_title',
     'section_path': "RES - Sécurité juridique - Garantie contre les changements de position de l'administration fiscale - Délai de dépôt d'une demande de rescrit relatif aux jeunes entreprises innovantes (JEI) - (LPF, art. L 80 B, 4°)"},
    {'rank': 2,
     'score': 10.908,
     'boi_reference': 'BOI-TVA-SECT-80-60-10-50-20120912',
     'chunk_id': '882-PGP__document_title',
     'ch

## 6. Conclusion actuelle

Le double étage montre enfin quelque chose de propre:
- retrieval documentaire BOI: oui, crédible
- retrieval de passages: encore à stabiliser, mais maintenant inspectable
- prochaine étape: améliorer l'ordering local et préparer la gate d'abstention
